In [4]:
import torch
from torch import nn
import math

In [36]:
class InputEmbedding(nn.Module):
    def __init__(self,d_model:int , vocab_size:int)->None:
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding=nn.Embedding(vocab_size,d_model)
        print(f'Embedding shape :{self.embedding.weight.shape}')
    def forward(self,x):
        # (batch,seq_len) ---> (batch,seq_len,d_model)
        return self.embedding(x)*math.sqrt(self.d_model)

In [38]:
embedding = InputEmbedding(d_model = 512 , vocab_size = 65)
x = torch.tensor([[1,2,5,8],[2,5,2,8],[3,6,3,7],[5,6,4,9]])
output =embedding(x)
print(x.shape)
print(output.shape)


Embedding shape :torch.Size([65, 512])
torch.Size([4, 4])
torch.Size([4, 4, 512])


### for even idx
#### PE(pos, 2i)   = sin(pos / 10000^(2i / d_model)) 
### for odd idx
#### PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))

In [49]:
class PositionalEncoding(nn.Module):
    def __init__(self,d_model:int , seq_len:int ,dropout:float)->None:
        super().__init__()

        # embedding dimension size
        self.d_model = d_model

        # maximum sequence length
        self.seq_len = seq_len

        # dropout layer applied after adding positional encoding
        self.dropout = nn.Dropout(dropout)

        # create positional encoding matrix of shape (seq_len, d_model)
        pe = torch.zeros(seq_len,d_model)

        # create position indices tensor of shape (seq_len,1)
        # example:
        # [[0],
        #  [1],
        #  [2]]
        position = torch.arange(0,seq_len).unsqueeze(1)

        # convert integer tensor to float tensor because
        # sin() and cos() operations work on float tensors
        position = position.float()

        # create denominator term for positional encoding
        # shape -> (d_model/2,)
        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        # apply sin to even indices: 0,2,4...
        # broadcasting:
        # (seq_len,1) * (d_model/2,)
        # -> (seq_len,d_model/2)
        pe[:,::2]= torch.sin(position*div_term)

        # apply cos to odd indices: 1,3,5...
        pe[:,1::2] = torch.cos(position*div_term)

        # add batch dimension
        # shape: (seq_len,d_model) -> (1,seq_len,d_model)
        pe = pe.unsqueeze(0)

        # save positional encoding inside model
        # buffer is saved with model but not trained
        self.register_buffer('pe',pe)
    def forward(self,x):
        x = x+(self.pe[:,:x.shape[1],:]).requires_grad_(False)
        return self.dropout(x)

In [51]:
positionalEncoding = PositionalEncoding(d_model = 512,seq_len=4,dropout=1e-9)
output= positionalEncoding(output)

In [ ]:
class MultiheadAttention(nn.Module):

    def __init__(self, d_model: int, h: int, dropout: float):
        super().__init__()

        # Total embedding dimension
        # Example: 512
        self.d_model = d_model

        # Number of attention heads
        # Example: 8 heads
        self.h = h

        # Dropout layer used after softmax attention
        self.dropout = nn.Dropout(dropout)

        # Ensure embedding dimension can be equally split among all heads
        # Example:
        # 512 % 8 = 0 
        assert d_model % h == 0, 'd_model not divisible by h'

        # Dimension handled by each head
        # Example:
        # 512 // 8 = 64
        self.d_k = d_model // h

        # Linear layer to generate Query vectors
        # (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        self.w_q = nn.Linear(d_model, d_model, bias=False)

        # Linear layer to generate Key vectors
        self.w_k = nn.Linear(d_model, d_model, bias=False)

        # Linear layer to generate Value vectors
        self.w_v = nn.Linear(d_model, d_model, bias=False)

        # Final output projection layer after combining all heads
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):

        # Size of vector handled by each head
        # query.shape = (batch, h, seq_len, d_k)
        d_k = query.shape[-1]
        # Calculate similarity scores between query and key
        # key.transpose(-2,-1):
        # (batch,h,seq_len,d_k) -> (batch,h,d_k,seq_len)
        # Matrix multiplication:
        # (batch,h,seq_len,d_k) @ (batch,h,d_k,seq_len)=(batch,h,seq_len,seq_len)
        # Divide by sqrt(d_k) to stabilize gradients
        attention_score = (
            query @ key.transpose(-2, -1)) / math.sqrt(d_k)
        # Apply mask if provided
        # Used in decoder to hide future words
        if mask is not None:
            # Wherever mask == 0
            # fill attention score with very small value
            # so softmax makes probability ≈ 0
            attention_score = attention_score.masked_fill(mask == 0,-1e9)
        # Convert scores into probabilities
        # Sum across last dimension becomes 1
        attention_score = attention_score.softmax(dim=-1)
        # Apply dropout on attention probabilities
        if dropout is not None:
            attention_score = dropout(attention_score)
        # Multiply attention weights with values
        # (batch,h,seq_len,seq_len) @(batch,h,seq_len,d_k)=(batch,h,seq_len,d_k)
        # Returns:
        # 1. final attention output
        # 2. attention map
        return (attention_score @ value), attention_score

    def forward(self, q, k, v, mask):
        # Create Query matrix
        # (batch, seq_len, d_model)->(batch, seq_len, d_model)
        query = self.w_q(q)
        # Create Key matrix
        key = self.w_k(k)
        # Create Value matrix
        value = self.w_v(v)
        # Split query into multiple heads
        # (batch, seq_len, d_model)-> (batch, seq_len, h, d_k)-> (batch, h, seq_len, d_k)
        query = query.view(
            query.shape[0],
            query.shape[1],
            self.h,
            self.d_k
        ).transpose(1, 2)
        # Split key into multiple heads
        key = key.view(
            key.shape[0],
            key.shape[1],
            self.h,
            self.d_k
        ).transpose(1, 2)
        # Split value into multiple heads
        value = value.view(
            value.shape[0],
            value.shape[1],
            self.h,
            self.d_k
        ).transpose(1, 2)
        # Calculate scaled dot-product attention
        x, attention_score = MultiheadAttention.attention(
            query,
            key,
            value,
            mask,
            self.dropout
        )
        # Combine all heads back together
        # (batch,h,seq_len,d_k) ->(batch,seq_len,h,d_k)
        x = x.transpose(1, 2)
        # contiguous() makes tensor memory continuous
        # required before view()
        # Final reshape:
        # (batch,seq_len,h,d_k)->(batch,seq_len,d_model)
        x = x.contiguous().view(
            x.shape[0],
            -1,
            self.h * self.d_k
        )
        # Final linear projection
        # (batch,seq_len,d_model)->(batch,seq_len,d_model)

        return self.w_o(x)